# Teks — Fine-tune transformer untuk regresi / ordinal / skor esai (k-fold + OOF)
**Kapan dipakai:** target berupa ANGKA dari teks: rating bintang, skor esai, tingkat keparahan,
harga dari deskripsi, kemiripan dua kalimat (STS, isi `TEXT_B`). `regresi_teks` (TF-IDF+Ridge)
= jalur cepat; ini jalur skor tinggi dengan GPU.

**Isi:** model buatan sendiri (`MeanPoolRegressor`: transformer + mean pooling + multi-sample
dropout + Linear(1)), target distandarkan per fold, loss MSE (alt Huber), prediksi di-clip ke
rentang train. Target bulat berurutan (1..5) → ambang ordinal dicari dari OOF untuk QWK.

**Alur:** Konfigurasi → baca → k-fold → latih tiap fold → OOF (RMSE, Pearson, QWK) → ambang
ordinal → simpan OOF → submission → validator.

**Saat lomba yang diubah:** sel Konfigurasi. `OUTPUT_MODE`: `"angka"` (kontinu) atau
`"ordinal"` (bilangan bulat lewat ambang OOF). Waktu mepet: `FOLDS_TO_RUN = [0]`.

**Data demo:** Yelp (`regression`, Inggris, bintang 1-5, 6.000 baris).

In [ ]:
import html
import os
import time

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score, mean_squared_error
from sklearn.model_selection import KFold
from torch import nn
from torch.nn.functional import huber_loss, mse_loss
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/regression"         # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/yelp_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/yelp_test.csv"                      # GANTI saat lomba
ANSWER_CSV = f"{DATA_FOLDER}/yelp_test_labels.csv"             # HAPUS saat lomba (kunci demo)
OUTPUT_FOLDER = "../../../outputs/text/finetune_regresi"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"
ID_COL = "id"                    # GANTI saat lomba
TEXT_A = "text"                  # GANTI saat lomba
TEXT_B = None                    # GANTI: kalimat kedua untuk STS / pasangan, None = teks tunggal
TARGET = "stars"                 # GANTI saat lomba
SUBMIT_COL = TARGET
OUTPUT_MODE = "ordinal"          # "angka" = kontinu (RMSE), "ordinal" = bulat via ambang (QWK)
N_FOLDS = 5
SPLIT_SEED = 42
NOTEBOOK = "finetune_regresi"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"

In [ ]:
PRETRAINED = "../../../pretrained/distilbert_en"      # GANTI: demo Inggris
# PRETRAINED = "../../../pretrained/indobert"         # teks Indonesia
# PRETRAINED = "../../../pretrained/xlmr_base"        # campur bahasa
MODEL_NAME = "reg_distilbert"    # GANTI tiap variasi (nama file OOF)
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 3e-5
HEAD_LR = 1e-3                   # kepala baru belajar lebih cepat dari backbone
WARMUP = 0.1
DROPOUT_RATES = [0.1, 0.2, 0.3, 0.4, 0.5]   # multi-sample dropout: rata-rata 5 kepala dropout
LOSS_NAME = "mse"
# LOSS_NAME = "huber"            # target berpencilan / label berisik
FOLDS_TO_RUN = None
SEED = 42

## Metrik
RMSE (makin kecil makin baik) untuk memilih epoch terbaik. QWK dihitung bila `OUTPUT_MODE`
ordinal; Pearson/Spearman untuk STS.

In [ ]:
def rmse(truth, predicted):
    return mean_squared_error(truth, predicted) ** 0.5


def qwk(truth, predicted):
    return cohen_kappa_score(truth, predicted, weights="quadratic")


print("mode:", OUTPUT_MODE, "| device:", DEVICE)

## Baca data

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)


def column_text(frame, column):
    if column is None:
        return None
    texts = frame[column].fillna("").astype(str)
    return texts.apply(html.unescape)


train_a, test_a = column_text(train, TEXT_A), column_text(test, TEXT_A)
train_b, test_b = column_text(train, TEXT_B), column_text(test, TEXT_B)
target = train[TARGET].to_numpy().astype("float32")
LOW, HIGH = target.min(), target.max()
print("train:", train.shape, "| test:", test.shape, "| target min/maks:", LOW, HIGH)
target_summary = train[TARGET].describe()
print(target_summary.round(3).to_dict())

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist(), sample[SUBMIT_COL].dtype)

## Tokenisasi dan split

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)


def encode_rows(texts_a, texts_b):
    second = None
    if texts_b is not None:
        second = list(texts_b)
    encoded = tokenizer(list(texts_a), second, truncation=True, max_length=MAX_LENGTH)
    rows = []
    for number in range(len(encoded["input_ids"])):
        row = {}
        for key in encoded.keys():
            row[key] = encoded[key][number]
        rows.append(row)
    return rows


train_rows = encode_rows(train_a, train_b)
test_rows = encode_rows(test_a, test_b)
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED).split(train_rows))
lengths = []
for row in train_rows[:2000]:
    lengths.append(len(row["input_ids"]))
print("token p50/p95:", np.percentile(lengths, [50, 95]), "| fold:", len(folds))

## Model buatan sendiri
Mean pooling (rata-rata semua token, bukan hanya [CLS]) lebih stabil untuk regresi. Multi-sample
dropout: kepala yang sama dipakai di 5 versi dropout lalu dirata-rata (latih lebih stabil).

In [ ]:
class MeanPoolRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(PRETRAINED, local_files_only=True)
        self.drops = nn.ModuleList()
        for rate in DROPOUT_RATES:
            self.drops.append(nn.Dropout(rate))
        self.head = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, **rest):
        output = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **rest)
        hidden = output.last_hidden_state
        weights = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1)
        outputs = []
        for drop in self.drops:
            outputs.append(self.head(drop(pooled)))
        stacked = torch.stack(outputs).mean(dim=0)
        return stacked.squeeze(-1)


print("MeanPoolRegressor siap")

## Latih dan prediksi
Target distandarkan dengan rata-rata/simpangan data latih fold; prediksi dikembalikan ke skala
asli lalu di-clip ke [min, maks] train.

In [ ]:
def pick(rows, chosen):
    picked = []
    for number in chosen:
        picked.append(rows[number])
    return picked


def forward_values(model, rows):
    batch = tokenizer.pad(rows, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        return model(**batch).float()


def predict(model, rows, center, spread):
    model.eval()
    values = np.zeros(len(rows), dtype="float32")
    with torch.no_grad():
        for start in range(0, len(rows), BATCH_SIZE * 4):
            chosen = np.arange(start, min(start + BATCH_SIZE * 4, len(rows)))
            values[chosen] = forward_values(model, pick(rows, chosen)).cpu().numpy()
    return np.clip(values * spread + center, LOW, HIGH)


print("predict siap")

In [ ]:
def build_run(fit_index):
    torch.manual_seed(SEED)
    model = MeanPoolRegressor().to(DEVICE)
    groups = [{"params": model.backbone.parameters(), "lr": LEARNING_RATE},
              {"params": model.head.parameters(), "lr": HEAD_LR}]
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    steps = max(1, int(np.ceil(len(fit_index) / BATCH_SIZE))) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)
    center = float(target[fit_index].mean())
    spread = float(target[fit_index].std()) + 1e-6
    return {"model": model, "optimizer": optimizer, "scheduler": scheduler, "center": center,
            "spread": spread, "rng": np.random.default_rng(SEED)}


def compute_loss(predicted, wanted):
    if LOSS_NAME == "huber":
        return huber_loss(predicted, wanted, delta=1.0)
    return mse_loss(predicted, wanted)


print("build_run siap | loss", LOSS_NAME)

In [ ]:
def train_epoch(run, fit_index):
    model = run["model"]
    model.train()
    order = run["rng"].permutation(fit_index)
    for start in range(0, len(order), BATCH_SIZE):
        chosen = order[start:start + BATCH_SIZE]
        scaled = (target[chosen] - run["center"]) / run["spread"]
        wanted = torch.tensor(scaled, dtype=torch.float32).to(DEVICE)
        loss = compute_loss(forward_values(model, pick(train_rows, chosen)), wanted)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        run["optimizer"].step()
        run["scheduler"].step()
        run["optimizer"].zero_grad()


def snapshot(model):
    state = {}
    for key, tensor in model.state_dict().items():
        moved = tensor.detach().cpu()
        state[key] = moved.clone()
    return state


print("train_epoch siap")

In [ ]:
def train_fold(fit_index, val_index):
    run = build_run(fit_index)
    val_rows = pick(train_rows, val_index)
    best, state = np.inf, None
    for epoch in range(EPOCHS):
        began = time.time()
        train_epoch(run, fit_index)
        value = rmse(target[val_index], predict(run["model"], val_rows, run["center"],
                                                run["spread"]))
        if value < best:
            best, state = value, snapshot(run["model"])
        print(f"  epoch {epoch + 1}: RMSE {value:.4f} | {time.time() - began:.0f} dtk")
    run["model"].load_state_dict(state)
    val_pred = predict(run["model"], val_rows, run["center"], run["spread"])
    test_part = predict(run["model"], test_rows, run["center"], run["spread"])
    del run
    torch.cuda.empty_cache()
    return val_pred, test_part


print("train_fold siap")

## Latih k-fold

In [ ]:
folds_to_run = FOLDS_TO_RUN
if folds_to_run is None:
    folds_to_run = list(range(len(folds)))
oof = np.zeros(len(train), dtype="float32")
filled = np.zeros(len(train), dtype=bool)
test_pred = np.zeros(len(test), dtype="float32")
began = time.time()
for number in folds_to_run:
    fit_index, val_index = folds[number]
    val_pred, test_part = train_fold(fit_index, val_index)
    oof[val_index] = val_pred
    filled[val_index] = True
    test_pred += test_part / len(folds_to_run)
    print(f"fold {number}: RMSE {rmse(target[val_index], val_pred):.4f}")
total_seconds = round(time.time() - began, 1)
oof_rmse = rmse(target[filled], oof[filled])
print(f"RMSE OOF {oof_rmse:.4f} | Pearson {pearsonr(target[filled], oof[filled])[0]:.4f} | "
      f"Spearman {spearmanr(target[filled], oof[filled])[0]:.4f} | {total_seconds / 60:.1f} menit")

## Ambang ordinal (hanya `OUTPUT_MODE = "ordinal"`)
Pembulatan biasa memakai batas x.5. Batas terbaik untuk QWK sering bergeser; tiap batas digeser
-0.5..+0.5 bergiliran (3 putaran) selama QWK OOF naik. Target harus bulat berurutan (mis. 1..5).

In [ ]:
def apply_cuts(values, cuts):
    return np.digitize(values, cuts) + int(LOW)


def fit_cuts(values, truth):
    cuts = np.arange(LOW + 0.5, HIGH)
    for _ in range(3):
        for number in range(len(cuts)):
            best_value, best_cut = qwk(truth, apply_cuts(values, cuts)), cuts[number]
            for delta in np.linspace(-0.5, 0.5, 21):
                trial = cuts.copy()
                trial[number] = best_cut + delta
                value = qwk(truth, apply_cuts(values, np.sort(trial)))
                if value > best_value:
                    best_value, cuts = value, np.sort(trial)
    return cuts


cuts = np.arange(LOW + 0.5, HIGH)
if OUTPUT_MODE == "ordinal":
    plain = qwk(target[filled], apply_cuts(oof[filled], cuts))
    cuts = fit_cuts(oof[filled], target[filled])
    tuned = qwk(target[filled], apply_cuts(oof[filled], cuts))
    print(f"QWK OOF bulat biasa {plain:.4f} | ambang OOF {tuned:.4f}")
    print("ambang:", cuts.round(3))

## Simpan OOF dan catat percobaan
Disimpan nilai kontinu (n_train), bukan hasil ambang, agar `ensemble_oof` bisa merata-rata.

In [ ]:
oof_score = oof_rmse
metric_name = "rmse"
if OUTPUT_MODE == "ordinal":
    oof_score = qwk(target[filled], apply_cuts(oof[filled], cuts))
    metric_name = "qwk"
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_pred)
    print("OOF dan test disimpan:", MODEL_NAME)
note = (f"{os.path.basename(PRETRAINED)} {LOSS_NAME} | epoch {EPOCHS} lr {LEARNING_RATE} "
        f"len {MAX_LENGTH} | {OUTPUT_MODE}")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": NOTEBOOK, "model": MODEL_NAME,
       "metrik": metric_name, "skor_cv": round(oof_score, 4), "detik": total_seconds,
       "catatan": note}
log_path = f"{OOF_FOLDER}/percobaan.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(3).to_string())

## Submission
Ordinal: bilangan bulat lewat ambang. Angka: nilai kontinu (sudah di-clip).

In [ ]:
final = test_pred
if OUTPUT_MODE == "ordinal":
    final = apply_cuts(test_pred, cuts)
pred_by_id = pd.Series(final, index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
pred_summary = submission[SUBMIT_COL].describe()
print(pred_summary.round(3).to_dict())
submission.head()

## Cek submission terhadap sample_submission
Versi ringkas validator `simpan_muat` #15–#16. Tipe kolom harus sama dengan sample (bulat vs
desimal). Harus `MASALAH: 0`.

In [ ]:
written = pd.read_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}")
problems = []
if written.columns.tolist() != sample.columns.tolist():
    problems.append("kolom / urutan kolom beda")
if written[ID_COL].tolist() != sample[ID_COL].tolist():
    problems.append("id / urutan id beda dari sample")
if written[SUBMIT_COL].isna().any():
    problems.append("ada prediksi kosong")
if written[SUBMIT_COL].dtype != sample[SUBMIT_COL].dtype:
    problems.append(f"tipe {written[SUBMIT_COL].dtype}, sample {sample[SUBMIT_COL].dtype}")
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth = truth.reindex(submission[ID_COL]).to_numpy()
print(f"RMSE test (kunci): {rmse(truth, submission[SUBMIT_COL]):.4f} | OOF {oof_rmse:.4f}")
if OUTPUT_MODE == "ordinal":
    print(f"QWK test (kunci): {qwk(truth, submission[SUBMIT_COL]):.4f} | OOF {oof_score:.4f}")